In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

from src.config import Configuration
CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/notebooks
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/app


In [2]:
# Runtime toggles
# - Set FORCE_CPU=True to disable CUDA even if available.
# - Set GPU_ID to pick a specific GPU index.
FORCE_CPU = True
GPU_ID = 0

In [3]:
"""
Nemotron ColEmbed VL 4B V2 — Video Embeddings for Sexism Classification
========================================================================
Model: nvidia/nemotron-colembed-vl-4b-v2
Paper: arxiv.org/abs/2602.03992
"""

import json
import os
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
MODEL_ID = "nvidia/nemotron-colembed-vl-4b-v2"

# ---------------------------------------------------------------------------
# Sexism-steering query
# ---------------------------------------------------------------------------
SEXISM_QUERY = (
    "Detect sexist, misogynistic, or gender-discriminatory content. "
    "This includes: explicit degradation or verbal abuse targeting women; "
    "objectification of women's bodies or appearance for entertainment; "
    "gender stereotypes restricting women to domestic, submissive, or "
    "decorative roles; misogynistic humor, irony, or sarcasm that normalizes "
    "treating women as inferior; portrayals of women as intellectually, "
    "professionally, or morally subordinate to men; and content that "
    "trivializes or glorifies harassment, assault, or gender-based "
    "discrimination. Represent this video for downstream classification "
    "into sexist or non-sexist content."
 )

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
@dataclass
class LocalConfig:
    videos_path: str
    output_dir: str
    num_frames: int = 8
    batch_size: int = 8
    video_extensions: Tuple[str, ...] = (".mp4", ".avi", ".mov", ".mkv", ".webm")

EmbedConfig = LocalConfig(
    videos_path=CONFIG.videos_path,
    output_dir=CONFIG.MODEL_PATH,
 )

# ---------------------------------------------------------------------------
# Device / model loading
# ---------------------------------------------------------------------------

def get_device() -> torch.device:
    if FORCE_CPU:
        return torch.device("cpu")
    if not torch.cuda.is_available():
        return torch.device("cpu")
    # Quick smoke test to avoid silent CPU fallback when CUDA is misconfigured.
    try:
        _ = (torch.randn(64, 64, device=f"cuda:{GPU_ID}") @ torch.randn(64, 64, device=f"cuda:{GPU_ID}")).sum().item()
        return torch.device(f"cuda:{GPU_ID}")
    except Exception as e:
        print("CUDA is_available=True but runtime test failed; using CPU:", e)
        return torch.device("cpu")


def _pick_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    try:
        if getattr(torch.cuda, "is_bf16_supported", lambda: False)():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16


def _first_param_device(model) -> str:
    try:
        for p in model.parameters():
            return str(p.device)
    except Exception:
        pass
    return "unknown"


def load_model(device: torch.device):
    """Load the model and ensure real GPU execution when CUDA is selected.

    Important: this model's `forward_images/forward_queries` implementation
    internally moves batches to `self.device`. If we load with `device_map="auto"`,
    `self.device` can end up as CPU and the whole forward runs on CPU even when
    a GPU exists. So on CUDA we explicitly move the full model to `cuda:{GPU_ID}`.
    """
    from transformers import AutoModel

    dtype = _pick_dtype(device)
    print(f"Loading {MODEL_ID} on {device} (dtype={dtype}) …")

    if device.type == "cuda":
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass

    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=dtype,
    ).eval()

    if device.type == "cuda":
        try:
            model = model.to(device)
        except Exception as e:
            print("Moving model to CUDA failed; falling back to CPU:", e)
            model = model.to("cpu")
    else:
        model = model.to(device)

    print("Model param device:", _first_param_device(model))
    try:
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    except Exception:
        pass
    return model


def _model_uses_cuda(model) -> bool:
    try:
        for p in model.parameters():
            if p.device.type == "cuda":
                return True
    except Exception:
        pass
    return False

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def list_dir_files(videos_path: str) -> Tuple[List[str], int]:
    p = Path(videos_path)
    files = sorted(
        str(f.resolve())
        for f in p.rglob("*")
        if f.suffix.lower() in EmbedConfig.video_extensions
    )
    return files, len(files)


def sample_frames(video_path: str, num_frames: int, device: torch.device) -> List[Image.Image]:
    try:
        from decord import VideoReader, cpu, gpu
    except Exception as e:
        raise ImportError("`decord` is required for frame extraction; pip install decord") from e

    ctx = cpu(0)
    if device.type == "cuda":
        try:
            ctx = gpu(GPU_ID)
        except Exception:
            ctx = cpu(0)

    vr = VideoReader(video_path, ctx=ctx)
    total = len(vr)
    if total <= 0:
        raise ValueError(f"Empty/corrupt video: {video_path}")
    indices = np.linspace(0, total - 1, num=num_frames, dtype=int).tolist()
    frames = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(f).convert("RGB") for f in frames]


def _ensure_tensor(x: torch.Tensor | List[torch.Tensor] | Tuple[torch.Tensor, ...]) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        return x
    if isinstance(x, (list, tuple)) and len(x) > 0 and isinstance(x[0], torch.Tensor):
        return torch.stack(list(x), dim=0)
    raise TypeError(f"Unexpected embedding type: {type(x)}")


def embed_frames(frames: List[Image.Image], model, batch_size: int) -> torch.Tensor:
    with torch.inference_mode():
        embs = model.forward_images(frames, batch_size=batch_size)
    embs = _ensure_tensor(embs)
    if embs.dim() == 3:
        embs = embs.mean(dim=1)
    return embs


def embed_query(query: str, model) -> torch.Tensor:
    with torch.inference_mode():
        q = model.forward_queries([query], batch_size=1)
    q = _ensure_tensor(q)
    if q.dim() == 3:
        q = q.mean(dim=1)
    if q.dim() == 1:
        q = q.unsqueeze(0)
    return q


def pool_video_embedding(frame_embeddings: torch.Tensor) -> torch.Tensor:
    return frame_embeddings.mean(dim=0)

# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def run_embedding_pipeline():
    device = get_device()
    os.makedirs(EmbedConfig.output_dir, exist_ok=True)
    files, n = list_dir_files(EmbedConfig.videos_path)
    if n == 0:
        raise FileNotFoundError(f"No video files found at: {EmbedConfig.videos_path}")
    print(f"Found {n} video files.")

    model = load_model(device)
    print(f"Model loaded. CUDA active = {_model_uses_cuda(model)}")

    print("Embedding sexism query …")
    query_embeddings = embed_query(SEXISM_QUERY, model)

    all_video_embeddings: list[np.ndarray] = []
    all_scores: list[float] = []
    meta: list[dict] = []

    for i, vpath in enumerate(tqdm(files, desc="Embedding videos")):
        try:
            frames = sample_frames(vpath, num_frames=EmbedConfig.num_frames, device=device)
            frame_embs = embed_frames(frames, model, batch_size=EmbedConfig.batch_size)
            video_emb = pool_video_embedding(frame_embs)
            with torch.inference_mode():
                score = model.get_scores(query_embeddings, [video_emb.unsqueeze(0)])
            score_val = float(score.reshape(-1)[0].item())

            all_video_embeddings.append(video_emb.float().detach().cpu().numpy())
            all_scores.append(score_val)
            meta.append({
                "index": i,
                "path": vpath,
                "filename": os.path.basename(vpath),
                "sexism_score": score_val,
                "num_frames_sampled": EmbedConfig.num_frames,
                "embedding_dim": int(all_video_embeddings[-1].shape[0]),
                "query": SEXISM_QUERY,
                "model": MODEL_ID,
                "device": str(device),
            })
        except Exception as exc:
            print(f"[ERROR] {os.path.basename(vpath)}: {exc}")
            all_video_embeddings.append(np.zeros(2560, dtype=np.float32))
            all_scores.append(float("nan"))
            meta.append({
                "index": i,
                "path": vpath,
                "filename": os.path.basename(vpath),
                "sexism_score": float("nan"),
                "num_frames_sampled": EmbedConfig.num_frames,
                "embedding_dim": 2560,
                "query": SEXISM_QUERY,
                "model": MODEL_ID,
                "device": str(device),
                "error": str(exc),
            })

    emb_matrix = np.stack(all_video_embeddings)
    scores_arr = np.array(all_scores, dtype=np.float32)

    emb_path = os.path.join(EmbedConfig.output_dir, "embeddings.npy")
    scores_path = os.path.join(EmbedConfig.output_dir, "sexism_scores.npy")
    meta_path = os.path.join(EmbedConfig.output_dir, "embeddings_meta.json")

    np.save(emb_path, emb_matrix)
    np.save(scores_path, scores_arr)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Embeddings  → {emb_path}  shape={emb_matrix.shape}")
    print(f"✓ Scores      → {scores_path}")
    print(f"✓ Metadata    → {meta_path}")
    return emb_matrix, scores_arr, meta

In [ ]:
print("Selected device:", get_device(), "cuda_available=", torch.cuda.is_available())
emb_matrix, scores_arr, meta = run_embedding_pipeline()
print("Pipeline completed successfully")

Selected device: cpu cuda_available= True
Found 674 video files.


/home/turbotowerlnx/miniconda3/envs/ALC_env_2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading nvidia/nemotron-colembed-vl-4b-v2 on cpu (dtype=torch.float32) …
